In [ ]:
import random
import pandas as pd
import re
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import ElasticNet, Ridge
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
import joblib
from sklearn.multioutput import MultiOutputRegressor
import tensorflow as tf

In [ ]:
import seaborn as sns

# Prepocessing data


In [ ]:
df_0=pd.read_csv("/content/drive/MyDrive/df_0.csv") #for validation
df_1=pd.read_csv("/content/drive/MyDrive/df_1.csv")
df_2=pd.read_csv("/content/drive/MyDrive/df_2.csv")
df_3=pd.read_csv("/content/drive/MyDrive/df_3.csv")
df_4=pd.read_csv("/content/drive/MyDrive/df_4.csv")

In [ ]:
dfs = [df_1, df_2, df_3, df_4]

In [ ]:
def preprocess_spectra(df):
  part1 = df.iloc[:, 56:1779:3]
  part2 = df.iloc[:, -4:]
  new_data = pd.concat([part1, part2], axis=1)

  return new_data

In [ ]:
preproc_spectra=[]
for df in dfs:
  df_i = preprocess_spectra(df)
  preproc_spectra.append(df_i)


In [ ]:
X_2=preproc_spectra[3].iloc[:, :575]

In [ ]:
def corr_mat(X):
  correlation_matrix = X.corr()
  corr_no_diag = correlation_matrix.mask(np.eye(len(correlation_matrix), dtype=bool))

  mask = np.triu(np.ones_like(corr_no_diag, dtype=bool))


  sns.heatmap(
      corr_no_diag,
      mask=mask,
      cmap='coolwarm',
      vmin=-1, vmax=1,
      center=0,
      square=True,
      linewidths=0,  # убираем линии между ячейками
      # чуть уменьшаем цветовую шкалу
      annot=False  # без аннотаций для ускорения
  )

  plt.show()
  np.fill_diagonal(correlation_matrix.values, 0)

  mean_corr = correlation_matrix.values.mean()
  print(f"Mean correlaion: {mean_corr:.3f}")

In [ ]:
correlation_matrix = X_2.corr()  # корреляция Пирсона по умолчанию
corr_no_diag = correlation_matrix.mask(np.eye(len(correlation_matrix), dtype=bool))


mask = np.triu(np.ones_like(corr_no_diag, dtype=bool))


sns.heatmap(
      corr_no_diag,
      mask=mask,
      cmap='coolwarm',
      vmin=-1, vmax=1,
      center=0,
      square=True,
      linewidths=0,
      annot=False
  )

plt.show()
np.fill_diagonal(correlation_matrix.values, 0)

mean_corr = correlation_matrix.values.mean()
print(f"Mean correlation {mean_corr:.3f}")

In [ ]:
threshold = 0.7

In [ ]:
weak_correlated_features = []
for col in correlation_matrix.columns:
    if (corr_no_diag[col].abs() < threshold).any():
        weak_correlated_features.append(col)


In [ ]:
weak_counts = (corr_no_diag.abs() < threshold).sum(axis=0)

weak_counts_sorted = weak_counts.sort_values(ascending=False)

In [ ]:
top_n = int(len(weak_counts) * 0.1773)
top_weak_features = weak_counts_sorted.index[:top_n].tolist()

In [ ]:
X_y_weak_correlation=[]

for preproc_spectrum in preproc_spectra:
  X = preproc_spectrum.iloc[:, :575]
  Y=preproc_spectrum.iloc[:, 575:]
  X_weak_correlation=X[top_weak_features]
  data = pd.concat([X_weak_correlation, Y], axis=1)
  X_y_weak_correlation.append(data)

In [ ]:
corr_mat(X_y_weak_correlation[0].iloc[:, :-4])

# BUILDING A MODEL BASED ON LOW-CORRELATED FEATURES

In [ ]:
model = RandomForestRegressor(random_state=42)

r2_all = []
mse_all = []

for i in range(4):

    test_df = X_y_weak_correlation[i]

    train_df = pd.concat([df for j, df in enumerate(X_y_weak_correlation) if j != i], ignore_index=True)


    X_train = train_df.iloc[:, :-4]
    y_train = train_df.iloc[:, -4:]

    X_test = test_df.iloc[:, :-4]
    y_test = test_df.iloc[:, -4:]


    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    r2 = r2_score(y_test, y_pred, multioutput='raw_values')
    mse = mean_squared_error(y_test, y_pred, multioutput='raw_values')

    r2_all.append(r2)
    mse_all.append(mse)

r2_all = np.array(r2_all)
mse_all = np.array(mse_all)

Y = train_df.iloc[:, -4:].values

for i in range(Y.shape[1]):
    print(f"Output {i+1}:")
    print(f"  R² scores:  {r2_all[:, i]} | Mean R²:  {np.mean(r2_all[:, i]):.4f}")
    print(f"  MSE scores: {mse_all[:, i]} | Mean MSE: {np.mean(mse_all[:, i]):.4f}")
    print()

training on all 4 datasets

In [ ]:
train_full_df = pd.concat([df for df in X_y_weak_correlation])
X_train_full = train_full_df.iloc[:, :-4]
y_train_full = train_full_df.iloc[:, -4:]

In [ ]:
rf = RandomForestRegressor()
rf.fit(X_train_full, y_train_full)


RandomForestRegressor(max_depth=15, max_features='sqrt', min_samples_leaf=3,
                      min_samples_split=5, n_estimators=300, oob_score=True,
                      random_state=42)

In [ ]:
df_0_preproc=preprocess_spectra(df_0)
X_val = df_0_preproc.iloc[:, :575]
Y_val=df_0_preproc.iloc[:, 575:]
X_weak_correlation_val=X_val[top_weak_features]

In [ ]:
y_pred_val= rf.predict(X_weak_correlation_val)
r2 = r2_score(Y_val, y_pred_val, multioutput='raw_values')
mse = mean_squared_error(Y_val, y_pred_val, multioutput='raw_values')
print('R2', r2)
print('mse', mse)

R2 [0.24172742 0.75525163 0.96287344 0.90522293]
mse [0.00062786 0.00182755 0.00210163 0.00534489]


XGBoost

In [ ]:
r2_all = []
mse_all = []

for i in range(4):
    test_df = X_y_weak_correlation[i]

    train_df = pd.concat([df for j, df in enumerate(X_y_weak_correlation) if j != i],
                         ignore_index=True)

    X_train = train_df.iloc[:, :-4]
    y_train = train_df.iloc[:, -4:]

    X_test = test_df.iloc[:, :-4]
    y_test = test_df.iloc[:, -4:]

    y_train_sub = y_train.iloc[:, 1:]

    base_model = XGBRegressor(
        objective="reg:squarederror",
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=42
    )
    model = MultiOutputRegressor(base_model)
    model.fit(X_train, y_train_sub)

    y_pred_sub = model.predict(X_test)
    y_pred_sub = np.maximum(y_pred_sub, 0)


    y_pred0 = 1 - np.sum(np.abs(y_pred_sub), axis=1, keepdims=True)

    y_pred = np.hstack([y_pred0, y_pred_sub])

    r2 = r2_score(y_test, y_pred, multioutput='raw_values')
    mse = mean_squared_error(y_test, y_pred, multioutput='raw_values')

    r2_all.append(r2)
    mse_all.append(mse)

r2_all = np.array(r2_all)
mse_all = np.array(mse_all)

Y = train_df.iloc[:, -4:].values
for i in range(Y.shape[1]):
    print(f"Output {i+1}:")
    print(f"  R² scores:  {r2_all[:, i]} | Mean R²:  {np.mean(r2_all[:, i]):.4f}")
    print(f"  MSE scores: {mse_all[:, i]} | Mean MSE: {np.mean(mse_all[:, i]):.4f}")
    print()


training on all 4 datasets

In [ ]:
base_model = XGBRegressor(
        objective="reg:squarederror",
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=42)
model = MultiOutputRegressor(base_model)
model.fit(X_train_full, y_train_full)

MultiOutputRegressor(estimator=XGBRegressor(base_score=None, booster=None,
                                            callbacks=None,
                                            colsample_bylevel=None,
                                            colsample_bynode=None,
                                            colsample_bytree=0.5, device=None,
                                            early_stopping_rounds=None,
                                            enable_categorical=False,
                                            eval_metric=None,
                                            feature_types=None,
                                            feature_weights=None, gamma=None,
                                            grow_policy=None,
                                            importance_type=None,
                                            interaction_constraints=None,
                                            learning_rate=None, max_bin=None,
                                            max_cat_threshold=None,
                                            max_cat_to_onehot=None,
                                            max_delta_step=None, max_depth=None,
                                            max_leaves=None,
                                            min_child_weight=None, missing=nan,
                                            monotone_constraints=None,
                                            multi_strategy=None,
                                            n_estimators=None, n_jobs=None,
                                            num_parallel_tree=None, ...))

In [ ]:
y_pred_val= model.predict(X_weak_correlation_val)

r2 = r2_score(Y_val, y_pred_val, multioutput='raw_values')
mse = mean_squared_error(Y_val, y_pred_val, multioutput='raw_values')
print('R2', r2)
print('mse', mse)

R2 [-0.09405434  0.8993947   0.9622956   0.9495802 ]
mse [0.0009059  0.00075122 0.00213434 0.00284339]


LOG-ratios

In [ ]:
def alr_transform(y):
    """Converts a composition into a log relationship (ALR)"""
    reference = y[:, -1]  # last component as a reference
    ratios = y[:, :-1] / reference[:, None]
    return np.log(ratios)

def inverse_alr(x):
    """Reverse transformation"""
    exp_x = np.exp(x)
    denominator = 1 + np.sum(exp_x, axis=1, keepdims=True)
    composition = np.hstack([exp_x, np.ones((len(x), 1))]) / denominator
    return composition

XGBoost

In [ ]:
base_model = XGBRegressor(
        objective="reg:squarederror",
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=42
    )
model_xgb= MultiOutputRegressor(base_model)
r2_all = []
mse_all = []

for i in range(4):
    test_df = X_y_weak_correlation[i]

    train_df = pd.concat([df for j, df in enumerate(X_y_weak_correlation) if j != i], ignore_index=True)

    X_train = train_df.iloc[:, :-4]
    y_train = train_df.iloc[:, -4:]
    y_train_alr = alr_transform(y_train.values)

    X_test = test_df.iloc[:, :-4]
    y_test = test_df.iloc[:, -4:]

    model_xgb.fit(X_train, y_train_alr)

    y_test_alr = model_xgb.predict(X_test)
    y_pred = inverse_alr(y_test_alr)

    r2 = r2_score(y_test, y_pred, multioutput='raw_values')
    mse = mean_squared_error(y_test, y_pred, multioutput='raw_values')

    r2_all.append(r2)
    mse_all.append(mse)

r2_all = np.array(r2_all)
mse_all = np.array(mse_all)

Y = train_df.iloc[:, -4:].values

for i in range(Y.shape[1]):
    print(f"Output {i+1}:")
    print(f"  R² scores:  {r2_all[:, i]} | Mean R²:  {np.mean(r2_all[:, i]):.4f}")
    print(f"  MSE scores: {mse_all[:, i]} | Mean MSE: {np.mean(mse_all[:, i]):.4f}")
    print()

Output 1:
  R² scores:  [-0.59611514  0.19168034  0.43893243 -0.42234345] | Mean R²:  -0.0970
  MSE scores: [0.00132161 0.0006693  0.00046457 0.00117773] | Mean MSE: 0.0009

Output 2:
  R² scores:  [0.65661615 0.83977137 0.81928768 0.76142025] | Mean R²:  0.7693
  MSE scores: [0.00256406 0.00119643 0.00134939 0.00178149] | Mean MSE: 0.0017

Output 3:
  R² scores:  [0.92549964 0.95914473 0.87915878 0.8778076 ] | Mean R²:  0.9104
  MSE scores: [0.00421725 0.0023127  0.00684048 0.00691696] | Mean MSE: 0.0051

Output 4:
  R² scores:  [0.9257525  0.96311407 0.90305208 0.93124917] | Mean R²:  0.9308
  MSE scores: [0.00418714 0.00208016 0.00546731 0.00387716] | Mean MSE: 0.0039



In [ ]:
y_train_alr_full = alr_transform(y_train_full.values)
model_xgb = MultiOutputRegressor(base_model)
model_xgb.fit(X_train_full, y_train_full)

MultiOutputRegressor(estimator=XGBRegressor(base_score=None, booster=None,
                                            callbacks=None,
                                            colsample_bylevel=None,
                                            colsample_bynode=None,
                                            colsample_bytree=0.9, device=None,
                                            early_stopping_rounds=None,
                                            enable_categorical=False,
                                            eval_metric=None,
                                            feature_types=None,
                                            feature_weights=None, gamma=None,
                                            grow_policy=None,
                                            importance_type=None,
                                            interaction_constraints=None,
                                            learning_rate=0.05, max_bin=None,
                                            max_cat_threshold=None,
                                            max_cat_to_onehot=None,
                                            max_delta_step=None, max_depth=6,
                                            max_leaves=None,
                                            min_child_weight=None, missing=nan,
                                            monotone_constraints=None,
                                            multi_strategy=None,
                                            n_estimators=500, n_jobs=None,
                                            num_parallel_tree=None, ...))

In [ ]:
y_pred_val= model_xgb.predict(X_weak_correlation_val)

r2 = r2_score(Y_val, y_pred_val, multioutput='raw_values')
mse = mean_squared_error(Y_val, y_pred_val, multioutput='raw_values')
print('R2', r2)
print('mse', mse)

R2 [0.23096251 0.9104213  0.96889716 0.9549866 ]
mse [0.00063678 0.00066889 0.00176064 0.0025385 ]


Random Forest

In [ ]:
model = RandomForestRegressor(random_state=42)

r2_all = []
mse_all = []

for i in range(4):
    test_df = X_y_weak_correlation[i]

    train_df = pd.concat([df for j, df in enumerate(X_y_weak_correlation) if j != i], ignore_index=True)

    X_train = train_df.iloc[:, :-4]
    y_train = train_df.iloc[:, -4:]
    y_train_alr = alr_transform(y_train.values)

    X_test = test_df.iloc[:, :-4]
    y_test = test_df.iloc[:, -4:]


    model.fit(X_train, y_train_alr)

    y_test_alr = model.predict(X_test)
    y_pred = inverse_alr(y_test_alr)

    r2 = r2_score(y_test, y_pred, multioutput='raw_values')
    mse = mean_squared_error(y_test, y_pred, multioutput='raw_values')

    r2_all.append(r2)
    mse_all.append(mse)

r2_all = np.array(r2_all)
mse_all = np.array(mse_all)

Y = train_df.iloc[:, -4:].values

for i in range(Y.shape[1]):
    print(f"Output {i+1}:")
    print(f"  R² scores:  {r2_all[:, i]} | Mean R²:  {np.mean(r2_all[:, i]):.4f}")
    print(f"  MSE scores: {mse_all[:, i]} | Mean MSE: {np.mean(mse_all[:, i]):.4f}")
    print()

In [ ]:
y_train_alr_full = alr_transform(y_train_full.values)
model_forest_log = RandomForestRegressor(random_state=42)
model_forest_log.fit(X_train_full, y_train_alr_full)

RandomForestRegressor(random_state=42)

In [ ]:
y_pred_val_alr= model_forest_log.predict(X_weak_correlation_val)
y_pred_val = inverse_alr(y_pred_val_alr)

r2 = r2_score(Y_val, y_pred_val, multioutput='raw_values')
mse = mean_squared_error(Y_val, y_pred_val, multioutput='raw_values')
print('R2', r2)
print('mse', mse)

R2 [0.06274016 0.72609994 0.93868339 0.89609705]
mse [0.00077607 0.00204522 0.00347096 0.00585954]


# CNN

In [ ]:
train_full_df = pd.concat([df for df in preproc_spectra])
X_train_full = train_full_df.iloc[:, :-4]
y_train_full = train_full_df.iloc[:, -4:]

In [ ]:
np.random.seed(42)
X = X_train_full.values.astype(np.float32)
y = y_train_full.values.astype(np.float32)
y = y / y.sum(axis=1, keepdims=True)  # normalize to distribution

# TARGET VARIABLE PREPARATION

# Protect against zeros (to avoid log(0) in future losses)
epsilon = 1e-6
y = np.clip(y, epsilon, 1 - 3 * epsilon)
y = y / y.sum(axis=1, keepdims=True)

# SPECTRA NORMALIZATION
scaler = StandardScaler()
X_flat = X.reshape(-1, 575)
X_scaled = scaler.fit_transform(X_flat).astype(np.float32)
X_3d = X_scaled.reshape(-1, 575, 1)  # (n_samples, 575, 1)


# TRAIN/VALIDATION SPLIT

X_train, X_val, y_train, y_val = train_test_split(
    X_3d, y, test_size=0.15, random_state=42
)

# MODEL BUILDING

def build_1d_cnn(input_length=575, n_outputs=4):
    model = tf.keras.Sequential([
        # Block 1
        tf.keras.layers.Conv1D(64, kernel_size=7, activation='relu', input_shape=(input_length, 1)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling1D(pool_size=2),

        # Block 2
        tf.keras.layers.Conv1D(128, kernel_size=5, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling1D(pool_size=2),

        # Block 3
        tf.keras.layers.Conv1D(256, kernel_size=3, activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling1D(pool_size=2),

        # Block 4
        tf.keras.layers.Conv1D(256, kernel_size=3, activation='relu'),
        tf.keras.layers.BatchNormalization(),

        # Reduce to feature vector
        tf.keras.layers.GlobalAveragePooling1D(),

        # Fully connected part
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.4),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dropout(0.3),

        # Output
        tf.keras.layers.Dense(n_outputs),
        tf.keras.layers.Softmax()  # ← ensures sum = 1
    ])
    return model

model = build_1d_cnn(input_length=575, n_outputs=4)


# COMPILATION AND CALLBACKS

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',  # works with soft labels!
    metrics=['mae']
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5, verbose=1)
]

# TRAINING

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)


# EVALUATION

print("\nEvaluation on validation set:")
y_pred = model.predict(X_val)

mae_per_component = mean_absolute_error(y_val, y_pred, multioutput='raw_values')
print(f"MAE per component: {mae_per_component}")
print(f"Average MAE: {mae_per_component.mean():.4f}")


In [ ]:
y_pred = model.predict(X_val)

In [ ]:
df_0_preproc=preprocess_spectra(df_0)
X_val = df_0_preproc.iloc[:, :575]
Y_val=df_0_preproc.iloc[:, 575:]

In [ ]:
model.save('ir_cnn_model_2.keras')

In [ ]:
X_val_scaled = scaler.transform(X_val)
X_val_3d = X_val_scaled.reshape(-1, 575, 1)

In [ ]:
y_pred_val_cnn = model.predict(X_val_3d)

In [ ]:
r2_per_component  = r2_score(Y_val, y_pred_val_cnn, multioutput='raw_values')  # массив длины 4
mse_per_component = mean_squared_error(Y_val, y_pred_val_cnn, multioutput='raw_values')

print("R2 per components", r2_per_component)
print("MSE per components:", mse_per_component)